# Interactive Agent Session: Chapter 2 — LangChain Corporate HR Knowledge Agent

**The Business Problem:** HR teams answer the same 50 questions daily. Employees wait hours for responses about PTO, benefits, and policies.

**The Solution:** A RAG (Retrieval-Augmented Generation) agent that ingests the full HR Handbook into a vector database, then answers any employee question grounded in official policy — zero hallucinations.

**Key Concepts:** Recursive Chunking → OpenAI Embeddings → ChromaDB Vector Store → Retrieval Chain

In [ ]:
import sys
!{sys.executable} -m pip install --quiet langchain langchain_openai langchain_community langchain_chroma chromadb python-dotenv

In [ ]:
%%capture rag_logs

import os
from dotenv import load_dotenv
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_chroma import Chroma
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain.chains import create_retrieval_chain
from langchain.chains.combine_documents import create_stuff_documents_chain
from langchain_core.prompts import ChatPromptTemplate

load_dotenv()

# ── THE FULL HR HANDBOOK (Enterprise-Grade Data) ──
HR_HANDBOOK = """
═══════════════════════════════════════════════════════════
TECHCORP GLOBAL — EMPLOYEE HANDBOOK v3.2 (2024-2026)
═══════════════════════════════════════════════════════════

SECTION 1: PAID TIME OFF (PTO)
Policy ID: HR-PTO-2026
- Full-time employees accrue 25 days of PTO per calendar year.
- Part-time employees accrue 12 days of PTO pro-rated.
- PTO requests must be submitted 2 weeks in advance via the HR portal.
- Carry-over limit: Maximum 5 unused days may roll into the next year.
- PTO is not paid out upon voluntary resignation within the first 6 months.

SECTION 2: SICK LEAVE
Policy ID: HR-SICK-2026
- All employees receive 10 paid sick days per year.
- A medical certificate is required for absences exceeding 3 consecutive days.
- Sick leave cannot be used as vacation and is non-transferable.
- Mental health days count as sick leave (max 3 per year without documentation).

SECTION 3: PARENTAL LEAVE
Policy ID: HR-PARENT-2026
- Primary caregivers: 16 weeks fully paid parental leave.
- Secondary caregivers: 6 weeks fully paid parental leave.
- Adoption leave: Same as biological parental leave policies.
- Parental leave must begin within 12 months of the child birth or adoption date.
- Employees may request part-time re-entry for up to 4 weeks after returning.

SECTION 4: REMOTE WORK POLICY
Policy ID: HR-REMOTE-2026
- All employees are eligible for hybrid work (3 days office, 2 days remote).
- Fully remote positions require VP-level approval and a signed remote work agreement.
- Remote employees must be available during core hours: 10:00 AM - 4:00 PM local time.
- The company provides a $500/year home office stipend for remote-eligible employees.
- International remote work requires Tax Compliance review (contact hr-tax@techcorp.com).

SECTION 5: COMPENSATION & BENEFITS
Policy ID: HR-COMP-2026
- Annual salary reviews are conducted in Q1 (January-March).
- Performance bonuses: Up to 15% of base salary for exceeding targets.
- Stock options vest over 4 years with a 1-year cliff.
- Health insurance: Company covers 90% of premiums for employees and 70% for dependents.
- 401(k) retirement plan: Company matches up to 6% of salary.
- Education reimbursement: Up to $5,000/year for job-related courses and certifications.

SECTION 6: BEREAVEMENT & EMERGENCY LEAVE
Policy ID: HR-BEREAVE-2026
- Immediate family (spouse, children, parents): 5 days fully paid.
- Extended family (siblings, grandparents, in-laws): 3 days fully paid.
- Jury duty: Full pay for up to 10 days per year.
- Natural disaster displacement: Up to 5 days paid emergency leave.

SECTION 7: CODE OF CONDUCT
Policy ID: HR-CONDUCT-2026
- Zero tolerance for harassment, discrimination, or retaliation.
- All complaints are investigated within 5 business days of filing.
- Confidential reporting available via ethics@techcorp.com or anonymous hotline.
- Social media policy: Employees must not disclose proprietary information online.
- Violation of conduct policies may result in immediate termination.
"""

USE_SIMULATION = not os.getenv("OPENAI_API_KEY")

# ── THE RAG PIPELINE ──
if not USE_SIMULATION:
    # 1. Chunking (split by sections for better retrieval)
    text_splitter = RecursiveCharacterTextSplitter(chunk_size=300, chunk_overlap=50, separators=["SECTION", "\n\n", "\n"])
    docs = [Document(page_content=HR_HANDBOOK, metadata={"source": "hr_handbook_v3.2.pdf", "version": "2026"})]
    chunks = text_splitter.split_documents(docs)

    # 2. Vector Store (Chroma)
    embeddings = OpenAIEmbeddings()
    vectorstore = Chroma.from_documents(documents=chunks, embedding=embeddings)
    retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

    # 3. LLM + Retrieval Chain
    llm = ChatOpenAI(model="gpt-4o-mini", max_tokens=200)
    system_prompt = (
        "You are TechCorp official HR Assistant. "
        "Answer ONLY using the provided context from the HR Handbook. "
        "If the answer is not in the context, say: That information is not in the handbook. "
        "Always cite the Policy ID when possible. "
        "Be concise and professional.\n\nContext: {context}"
    )
    prompt = ChatPromptTemplate.from_messages([("system", system_prompt), ("human", "{input}")])
    combine_docs_chain = create_stuff_documents_chain(llm, prompt)
    rag_chain = create_retrieval_chain(retriever, combine_docs_chain)

# ── RUN MULTIPLE QUERIES ──
QUERIES = [
    "How many vacation days do I get?",
    "What is the parental leave policy for fathers?",
    "Can I work fully remote?",
    "Does the company match 401k contributions?"
]

results = []
for q in QUERIES:
    if USE_SIMULATION:
        sim_answers = {
            "How many vacation days do I get?": "Full-time employees accrue 25 days of PTO per calendar year. Part-time employees accrue 12 days pro-rated. (Policy: HR-PTO-2026)",
            "What is the parental leave policy for fathers?": "Secondary caregivers receive 6 weeks fully paid parental leave. This applies equally to biological and adoptive parents. (Policy: HR-PARENT-2026)",
            "Can I work fully remote?": "Fully remote positions require VP-level approval and a signed remote work agreement. All employees are eligible for hybrid (3 office / 2 remote). (Policy: HR-REMOTE-2026)",
            "Does the company match 401k contributions?": "Yes, the company matches up to 6% of your salary in the 401(k) retirement plan. (Policy: HR-COMP-2026)"
        }
        results.append({"query": q, "answer": sim_answers.get(q, "N/A")})
    else:
        res = rag_chain.invoke({"input": q})
        results.append({"query": q, "answer": res["answer"]})

In [ ]:
from IPython.display import display, HTML

msgs_html = ""
for r in results:
    msgs_html += '<div style="margin-bottom:25px; border-bottom:1px solid #1a1e2e; padding-bottom:25px;">'
    msgs_html += '<div style="display:flex; align-items:center; gap:8px; margin-bottom:12px;">'
    msgs_html += '<div style="width:28px; height:28px; border-radius:50%; background:#6366f1; display:flex; align-items:center; justify-content:center; font-size:12px;">👤</div>'
    msgs_html += '<div style="font-size:14px; color:#e2e8f0;">' + r["query"] + '</div>'
    msgs_html += '</div>'
    msgs_html += '<div style="display:flex; align-items:flex-start; gap:8px; margin-left:36px;">'
    msgs_html += '<div style="width:28px; height:28px; border-radius:50%; background:#22c55e; display:flex; align-items:center; justify-content:center; font-size:12px; flex-shrink:0;">🤖</div>'
    msgs_html += '<div style="font-size:13px; color:#94a3b8; line-height:1.6;">' + r["answer"] + '</div>'
    msgs_html += '</div></div>'

html = '<style>@import url("https://fonts.googleapis.com/css2?family=Inter:wght@400;700&family=Unbounded:wght@800&display=swap");</style>'
html += '<div style="max-width:900px; margin:30px auto; font-family:Inter,sans-serif; background:#0a0a0f; padding:45px; border-radius:40px; border:1px solid #1a1e2e; box-shadow:0 50px 100px rgba(0,0,0,0.8);">'
html += '<div style="display:flex; align-items:center; justify-content:space-between; margin-bottom:35px; border-bottom:1px solid #1a1e2e; padding-bottom:25px;">'
html += '<div style="font-family:Unbounded,sans-serif; font-size:16px; color:#6366f1; letter-spacing:1px;">HR_KNOWLEDGE_AGENT</div>'
html += '<div style="font-size:10px; background:#22c55e22; color:#22c55e; padding:4px 12px; border-radius:50px;">RAG_PIPELINE_ACTIVE</div>'
html += '</div>'
html += '<div style="font-size:10px; color:#334155; margin-bottom:25px;">SOURCE: hr_handbook_v3.2.pdf • 7 SECTIONS • CHROMA_VECTORSTORE</div>'
html += msgs_html
html += '</div>'
display(HTML(html))